In [ ]:
from ipywidgets import interact
import numpy as np
import matplotlib.pyplot as plt

def calcWeightChange(p,Ca,rate,gammaP,gammaD,thetaP,thetaD):
    #from the Graupner and Brunel paper
    #rate = 1/tau 
    pStar = 0.5
    revP = (1-p)
    fp = -p*revP*(pStar-p)
    potentiation = gammaP*revP*float(Ca>thetaP)
    depression = -gammaD*p*float(thetaD<Ca<=thetaP)
    Noise = 0
    return rate*fp+potentiation+depression+Noise

def runPlasticitySim(f,phase,thetaP,thetaD,gammaP,gammaD,rate,runTime=1000,TS=0.05):
    timeVec = np.arange(0,runTime,TS)

    #Ca = (0.5*(1+np.sin(2*3.14159*f*timeVec)))**(100+period)
    sin1 = np.sin(2*3.14159*f*timeVec)
    sin2 = np.sin(2*3.14159*f*(timeVec-phase))
    maxVal1 = np.sin(2*3.14159*f*phase)
    Ca = np.logical_and(sin1>0,sin2<0).astype(np.float32)*(1-sin1/maxVal1)
    #Ca[-int(100/TS)::] = 0


    p=0
    ps = np.zeros_like(Ca)
    for i,t in enumerate(timeVec):
        dp = calcWeightChange(p,Ca[i],rate,gammaP,gammaD,thetaP,thetaD)
        p += TS*dp 
        ps[i] = p
    
    return timeVec,ps,Ca

def plotGBrule(freq,phase,thetaP,thetaD,gammaP,gammaD,rate):
    
    f=freq/1000
    timeVec,ps,Ca = runPlasticitySim(f,phase,thetaP,thetaD,gammaP,gammaD,rate,TS=0.025)
    fig,ax = plt.subplots(2,1,figsize=(10,4),dpi=150,sharex=True)
    
    ax[0].plot(timeVec,Ca,label="Ca")
    # ax[0].plot(timeVec,sin1,label="sin1")
    # ax[0].plot(timeVec,sin2,label="sin2")
    ax[0].legend()
    ax[0].axhline(thetaD,ls="--",c="r")
    ax[0].axhline(thetaP,ls="-",c="b")
    ax[0].set_title("Ca")
    ax[0].set_ylabel("Ca Conc (A.U.)")
    ax[1].plot(timeVec,ps)
    ax[1].set_ylim(-0.01,1.01)
    ax[1].set_title("Synaptic Efficacy")
    ax[1].set_ylabel("Synaptic \nEfficacy (A.U.)")
    ax[1].set_xlabel("Time (ms)")

    


interact(plotGBrule,freq=(1,100),phase = (0,10,0.1),
         thetaP=(0,1,0.01),thetaD=(0,1,0.01),gammaP=(0,1,0.01),gammaD=(0,1,0.01),rate=(0,0.1,0.002))

interactive(children=(IntSlider(value=50, description='freq', min=1), FloatSlider(value=5.0, description='phas…

<function __main__.plotGBrule(freq, phase, thetaP, thetaD, gammaP, gammaD, rate)>

In [27]:
plasticityParams = {"thetaP":None,"thetaD":None,"gammaP":None,"gammaD":None,"rate":None}
#plot freq vs final mean value
def plotFreqResp(phase,thetaP,thetaD,gammaP,gammaD,rate):
    global plasticityParams
    plasticityParams["thetaP"],plasticityParams["thetaD"] = thetaP,thetaD
    plasticityParams["gammaP"],plasticityParams["gammaD"] = gammaP,gammaD
    plasticityParams["rate"] = rate
    startFreq,endFreq = 1,200 #Hz
    startFreq,endFreq = startFreq/1000,endFreq/1000 #convert to mHz
    freqs = np.geomspace(startFreq,endFreq,9)
    vals = []
    TS = 0.04
    fig,ax = plt.subplots(2,1,figsize = (8,8),dpi=120,sharey=True)

    for freq in freqs:
        timeVec,ps,Ca = runPlasticitySim(freq,phase,thetaP,thetaD,gammaP,gammaD,rate,runTime=1500,TS=TS)
        val = np.mean(ps[-int(200/TS)::])
        vals.append(val)
        ax[1].plot(timeVec,ps)

    vals = np.array(vals)
    ax[0].plot(freqs,vals)
    ax[0].set_ylim(-0.01,1.01)
    ax[0].set_xscale("log")
    
interact(plotFreqResp,phase = (0,3,0.1),
         thetaP=(0,1,0.01),thetaD=(0,1,0.01),gammaP=(0,0.2,0.002),gammaD=(0,0.2,0.002),rate=(0,0.1,0.002))

interactive(children=(FloatSlider(value=1.0, description='phase', max=3.0), FloatSlider(value=0.0, description…

<function __main__.plotFreqResp(phase, thetaP, thetaD, gammaP, gammaD, rate)>

In [28]:
plasticityParams

{'thetaP': 0.68,
 'thetaD': 0.33,
 'gammaP': 0.12,
 'gammaD': 0.07800000000000001,
 'rate': 0.05}